# TP2 : enrichissement du corpus CAMille, Le Soir en 1960

Ce notebook rassemble mes analyses des 100 pages du Soir de 1960. J'y cherche les mots-clés de l'année, je construis un nuage de mots, je repère les personnes, lieux et organisations les plus cités, puis j'étudie le sentiment exprimé dans dix phrases.

In [1]:
import os
import re
from collections import Counter, defaultdict

import yake
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from IPython.display import Image
import spacy
import pandas as pd
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

c:\Users\DERRICK\tac\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Choix de l'année et des fichiers

In [4]:
year = 1960
txt_path = "../../data/txt/"
temp_path = "../../data/tmp"

files = sorted([f for f in os.listdir(txt_path) if f.endswith('.txt') and f"_{year}-" in f])
len(files)

100

## 1. Les mots-clés de l'année avec YAKE

YAKE repère les expressions importantes de chaque page. Je garde celles de deux mots, comme « Congo belge » ou « Nations Unies », puis je regarde lesquelles reviennent le plus souvent au fil de l'année.

In [5]:
kw_extractor = yake.KeywordExtractor(lan="fr", top=50)

# Mots-clés de deux mots de chaque page de l'année
for f in files:
    with open(os.path.join(txt_path, f), 'r', encoding='utf-8') as fh:
        text = fh.read()
    keywords = kw_extractor.extract_keywords(text)
    kept = []
    for kw, score in keywords:
        words = kw.split()
        if len(words) == 2:
            kept.append(kw)
    print(f"{f} mentions these keywords: {', '.join(kept)}...")

KB_JB838_1960-01-10_01-00018.txt mentions these keywords: Agence Rossel, l’Agence Rossel, Ecrire Agence, Faire offre, Rossel num, IMPORTANTE SOCIETE, Importante firme, BRUXELLES CHERCHE, Importante société, Firme Importante, curriculum vitae, offre manuscrite, vltae Agence...
KB_JB838_1960-01-12_01-00011.txt mentions these keywords: Van den, Van der, classements officiels, Coq Anglais, Van Houtte, Country Club, Léopold Club, Van Blesebroeck, Astrld Club, Van Belle, Baudouin Club...
KB_JB838_1960-01-13_01-00009.txt mentions these keywords: VAN LOOY, Van Stcenbergen, Rik Van, DIVISION III, Fort Jaco, Grand Prix, grand temps, Dimanche division, dimanche dernier...
KB_JB838_1960-01-21_01-00014.txt mentions these keywords: chemise lavée, CHEMISES LAVÉES, Office Propriétaires, Propriétaires Tél, WATERLOO TéL, maison rapport, belle maison, PRIX TARIF, francs Office, bonne maison...
KB_JB838_1960-01-22_01-00015.txt mentions these keywords: LOUER App, Pastilles Antibiotiques, FER FORGE, spac. a

In [6]:
# Les expressions qui reviennent dans le plus grand nombre de pages
year_keywords = Counter()
for f in files:
    with open(os.path.join(txt_path, f), 'r', encoding='utf-8') as fh:
        text = fh.read()
    for kw, score in kw_extractor.extract_keywords(text):
        if len(kw.split()) == 2:
            year_keywords[kw] += 1

year_keywords.most_common(30)

[('Agence Rossel', 17),
 ('Ecrire Agence', 11),
 ('Rossel num', 9),
 ('l’Agence Rossel', 6),
 ('Faire offre', 6),
 ('Congo belge', 5),
 ('New York', 5),
 ('demeurant rue', 5),
 ('Bruxelles Tél', 5),
 ('curriculum vitae', 4),
 ('Van den', 4),
 ('Rik Van', 4),
 ('Grand Prix', 4),
 ('JOURNAL QUOTIDIEN', 4),
 ('habitant rue', 4),
 ('Bruxelles dem', 4),
 ('Van Looy', 4),
 ('Nations Unies', 4),
 ('offre manuscrite', 3),
 ('DIVISION III', 3),
 ('Propriétaires Tél', 3),
 ('rue Van', 3),
 ('Opel Rek', 3),
 ('SOIR MERCREDI', 3),
 ('d’une fracture', 3),
 ('Dem. fem', 3),
 ('l’AGENCE ROSSEL', 3),
 ('domicilié rue', 3),
 ('lundi matin', 3),
 ('gouvernement belge', 3)]